# Modul 15: TensorFlow-Grundlagen und dichte Keras-Modelle

    **Notebooktyp:** Übungs- und Bewertungsnotebook  
    **Vorlesungen dieses Moduls:** TensorFlow Grundlagen, Dichte Keras-Modelle  
    **Erwarteter Schwierigkeitsgrad:** Mittlere bis fortgeschrittene Framework-Anwendung  
    **Orientierungszeit:** etwa 130 bis 180 Minuten

    ## Überblick

    Sie arbeiten mit TensorFlow-Tensoren, automatischen Gradienten und tf.data. Danach bauen, trainieren, bewerten und speichern Sie ein kleines dichtes Keras-Modell für eine binäre Klassifikationsaufgabe.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_15A_20260723.ipynb`
- `ML Für Anfänger - Record_Module_15B_20260723.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - Tensorformen, Datentypen, NumPy-Konvertierung und Broadcasting sicher handhaben.
- Gradienten mit GradientTape berechnen und mit einer manuellen Ableitung vergleichen.
- tf.data-Datasets mit Mapping, Shuffling, Batching und Prefetching vorbereiten.
- Sequential-Modelle mit zur Aufgabe passenden Eingabe- und Ausgabeschichten erstellen.
- Loss, Optimierer und Metriken passend zur binären Klassifikation wählen.
- Trainingsverläufe, Vorhersagewahrscheinlichkeiten und Generalisierung interpretieren.
- Regularisierung, Baseline-Vergleich sowie Speichern und Laden in einem Mini-Projekt verbinden.

    ## Bewertete Fähigkeiten

    - TensorFlow-Tensoren, Broadcasting und GradientTape
- tf.data mit map, shuffle, batch und prefetch
- Keras Sequential, compile, fit, evaluate und predict
- Early Stopping, L2-Regularisierung und Dropout
- Baseline-Vergleich und reproduzierbares Speichern im Keras-Format

## Arbeitsanweisungen

Bearbeiten Sie die Aufgaben in der angegebenen Reihenfolge. Schreiben Sie Ihren Code ausschließlich in die klar markierten Arbeitszellen. Ergänzen Sie nach jeder Aufgabe eine kurze fachliche Reflexion. Verwenden Sie das Testset nicht für Modellwahl oder Hyperparameterentscheidungen, sofern die Aufgabe dies nicht ausdrücklich als abschließenden Schritt verlangt.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# TensorFlow ist in Google Colab üblicherweise bereits verfügbar.
# Der Fallback installiert nur dann eine CPU-Version, wenn der Import fehlt.
import os
import sys
import subprocess
import warnings
from pathlib import Path

try:
    import tensorflow as tf
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tensorflow-cpu"])
    import tensorflow as tf

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42
FAST_MODE = os.environ.get("COURSE_FAST", "0") == "1"
OFFLINE_MODE = os.environ.get("COURSE_OFFLINE", "0") == "1"

np.random.seed(RANDOM_SEED)
tf.keras.utils.set_random_seed(RANDOM_SEED)
warnings.filterwarnings("ignore", category=FutureWarning)

import tempfile

from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Ein kleiner tabellarischer Datensatz wird vollständig lokal erzeugt.
X_15, y_15 = make_classification(
    n_samples=900,
    n_features=12,
    n_informative=7,
    n_redundant=2,
    weights=[0.62, 0.38],
    class_sep=1.0,
    flip_y=0.06,
    random_state=RANDOM_SEED,
)
X_train_valid_15, X_test_15, y_train_valid_15, y_test_15 = train_test_split(
    X_15,
    y_15,
    test_size=0.20,
    stratify=y_15,
    random_state=RANDOM_SEED,
)
X_train_15, X_valid_15, y_train_15, y_valid_15 = train_test_split(
    X_train_valid_15,
    y_train_valid_15,
    test_size=0.25,
    stratify=y_train_valid_15,
    random_state=RANDOM_SEED,
)

# Die Standardisierung lernt ausschließlich aus dem Training.
scaler_15 = StandardScaler()
X_train_15 = scaler_15.fit_transform(X_train_15).astype("float32")
X_valid_15 = scaler_15.transform(X_valid_15).astype("float32")
X_test_15 = scaler_15.transform(X_test_15).astype("float32")
y_train_15 = y_train_15.astype("float32")
y_valid_15 = y_valid_15.astype("float32")
y_test_15 = y_test_15.astype("float32")

print("Train/Valid/Test:", X_train_15.shape, X_valid_15.shape, X_test_15.shape)

print("TensorFlow-Version:", tf.__version__)
print("Schneller Validierungsmodus:", FAST_MODE)


## Aufgabe 1: Tensoren, Broadcasting und automatische Gradienten

    Untersuchen Sie TensorFlow-Tensoren und kontrollieren Sie eine automatische Ableitung.

1. Erzeugen Sie einen Skalar, einen Vektor, eine Matrix und einen kleinen Bildstapel als `tf.Tensor`.
2. Geben Sie jeweils Form, Rang und Datentyp aus und konvertieren Sie die Matrix zurück nach NumPy.
3. Standardisieren Sie jede Spalte einer `3 x 2`-Matrix durch Broadcasting mit vorgegebenen Mittelwerten und Standardabweichungen.
4. Verwenden Sie `tf.GradientTape`, um für `loss(w) = mean((w*x - y)^2)` den Gradienten nach `w` zu berechnen.
5. Leiten Sie denselben Gradienten manuell ab und vergleichen Sie beide Werte numerisch.

> **Hinweis:** Prüfen Sie bei Broadcasting zuerst die Form der letzten Achse.

In [ ]:
feature_matrix = tf.constant(
    [[2.0, 10.0], [4.0, 14.0], [6.0, 18.0]],
    dtype=tf.float32,
)
feature_means = tf.constant([4.0, 14.0], dtype=tf.float32)
feature_stds = tf.constant([2.0, 4.0], dtype=tf.float32)

x_gradient = tf.constant([1.0, 2.0, 3.0], dtype=tf.float32)
y_gradient = tf.constant([2.0, 4.0, 5.0], dtype=tf.float32)

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 1

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Prüfen Sie bei Broadcasting zuerst die Form der letzten Achse.

## Aufgabe 2: Eine tf.data-Pipeline aufbauen

    Erstellen Sie eine reproduzierbare Eingabepipeline aus den Trainingsdaten.

1. Erzeugen Sie ein Dataset aus Merkmalen und Labels.
2. Schreiben Sie eine Mapping-Funktion, die die Merkmale leicht mit deterministischem Faktor skaliert und das Label in die Form `(1,)` bringt.
3. Mischen Sie nur die Trainingsdaten mit einem festen Seed.
4. Bilden Sie Batches der Größe 32 und verwenden Sie `prefetch(tf.data.AUTOTUNE)`.
5. Erstellen Sie getrennte Validierungs- und Test-Datasets ohne Shuffling.
6. Untersuchen Sie einen Batch und bestätigen Sie Formen und Datentypen.

> **Hinweis:** Erzeugen Sie erst einen funktionierenden Dataset-Batch und ergänzen Sie danach Prefetching.

In [ ]:
batch_size_15 = 32

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 2

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Erzeugen Sie erst einen funktionierenden Dataset-Batch und ergänzen Sie danach Prefetching.

## Aufgabe 3: Ein passendes Sequential-Modell definieren und kompilieren

    Erstellen Sie ein dichtes Keras-Modell für die vorbereitete binäre Klassifikation.

1. Verwenden Sie eine ausdrückliche Eingabeschicht für zwölf Merkmale.
2. Bauen Sie zwei kleine Dense-Schichten mit ReLU-Aktivierung.
3. Verwenden Sie genau ein Ausgabeneuron mit Sigmoid.
4. Kompilieren Sie das Modell mit Adam, binärer Kreuzentropie, Accuracy und AUC.
5. Geben Sie eine Modellzusammenfassung aus und prüfen Sie die Ausgabeform für einen Mini-Batch.
6. Begründen Sie, warum eine lineare Ausgabe mit MSE oder eine zehnklassige Softmax hier unpassend wäre.

> **Hinweis:** Leiten Sie die Zahl der Ausgabeneuronen direkt aus der Zielcodierung ab.

In [ ]:
# Speichern Sie das Modell unter dem Namen model_15, damit die
# folgenden Aufgaben es weiterverwenden können.

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 3

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Leiten Sie die Zahl der Ausgabeneuronen direkt aus der Zielcodierung ab.

## Aufgabe 4: Trainieren, Lernkurven prüfen und Vorhersagen interpretieren

    Trainieren und bewerten Sie `model_15`.

1. Verwenden Sie das Trainings- und Validierungs-Dataset aus Aufgabe 2.
2. Trainieren Sie mit `EarlyStopping`, überwachen Sie `val_loss` und stellen Sie die besten Gewichte wieder her.
3. Visualisieren Sie Loss, Accuracy und AUC für Training und Validierung.
4. Bewerten Sie das Modell genau einmal auf dem Test-Dataset.
5. Wandeln Sie Testwahrscheinlichkeiten mit Schwelle 0.5 in Klassen um und erstellen Sie eine Konfusionsmatrix.
6. Zeigen Sie fünf Beispiele mit ihrer Wahrscheinlichkeit, Vorhersage und wahrem Label.

> **Hinweis:** Nutzen Sie die Schlüssel von `history.history`, statt Metriknamen zu erraten.

In [ ]:
epochs_15 = 8 if FAST_MODE else 50

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 4

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Nutzen Sie die Schlüssel von `history.history`, statt Metriknamen zu erraten.

## Aufgabe 5: Integration: Regularisierung, Baseline und Modellartefakt

    Erstellen Sie ein zweites, reguliertes Modell und schließen Sie den Workflow reproduzierbar ab.

1. Trainieren Sie eine `DummyClassifier`-Baseline nur auf den Trainingsdaten und bewerten Sie sie auf der Validierung.
2. Erstellen Sie ein kleineres Keras-Modell mit L2-Regularisierung und Dropout.
3. Trainieren Sie es mit denselben Trainings- und Validierungsdaten sowie Early Stopping.
4. Vergleichen Sie Baseline, ursprüngliches Modell und reguliertes Modell mit Balanced Accuracy auf denselben Validierungsdaten.
5. Wählen Sie anhand der Validierung das bessere Keras-Modell und bewerten Sie dieses auf dem Testset.
6. Speichern Sie das ausgewählte Modell im `.keras`-Format in einem temporären Ordner, laden Sie es neu und bestätigen Sie identische Referenzvorhersagen.

> **Hinweis:** Wählen Sie das Modell auf der Validierung und berichten Sie den Testwert erst danach.

In [ ]:
regularized_epochs_15 = 8 if FAST_MODE else 50

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 5

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Wählen Sie das Modell auf der Validierung und berichten Sie den Testwert erst danach.

## Abschluss und Selbstkontrolle

Prüfen Sie vor der Abgabe, ob alle Arbeitszellen ausgefüllt sind, das Notebook von oben nach unten ohne unerwartete Fehler läuft, alle Diagramme beschriftet sind und jede Reflexion Ihre Beobachtungen sowie mindestens eine mögliche Fehlerquelle enthält.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.